# 베이스라인 모델 1 — CatBoost

로직은 [baseline_catboost.py](baseline_catboost.py)에 있고, 이 노트북은 그걸 불러와서 실행 + 결과 해석을 남긴다.

**왜 CatBoost부터인가**: 결측치 전처리가 없어도 된다. 수치형 NaN은 CatBoost가 내부적으로 "최솟값보다 작은 값"으로 취급해서 분기 방향을 학습하고, 범주형 NaN은 하나의 카테고리로 자동 처리한다. `asof_*` 이력 피처의 콜드스타트 결측(EDA 2절)을 사람이 손대지 않고 모델이 알아서 다루게 하는 접근.

**검증 방식**: rolling out-of-time. `season < 2024`로 학습하고 `season == 2024`로 검증한다.  "OOT 2024" 기준과 동일해서 우리 결과와 그 팀 결과를 직접 비교할 수 있다.

**피처**: `eda.py`에서 확정한 스키마를 그대로 재사용. 수치형(30) + 시계열형 중 순환 없는 것(`season`, `inning`, 수치로) + 이진형(7) + 범주형(3) + 시계열형 중 순환하는 것(`game_month`, `game_dayofweek`, 범주로) = 44개 피처. 범주형/이진형/순환 시계열형은 CatBoost 네이티브 카테고리 처리(`cat_features`)로 넘겨서 원핫 인코딩도 안 했다.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "eda"))

from baseline_catboost import *  # noqa: F403

print(f"피처 {len(FEATURES)}개, 그중 범주형(cat_features) {len(CAT_FEATURES)}개")
print("cat_features:", CAT_FEATURES)

피처 44개, 그중 범주형(cat_features) 12개
cat_features: ['top_bottom', 'game_type', 'pitcher_hand', 'batter_hand', 'runner_on_1b', 'runner_on_2b', 'runner_on_3b', 'base_state', 'pitcher_team_id', 'batter_team_id', 'game_month', 'game_dayofweek']


## 데이터 분할

In [2]:
df = load("train.csv")
train_df, valid_df = time_split(df)
print(f"train: {len(train_df):,}행 (season<{VALID_SEASON})")
print(f"valid: {len(valid_df):,}행 (season={VALID_SEASON})")
print(f"valid 타겟 평균(r): {valid_df[TARGET].mean():.4f}")

train: 1,221,585행 (season<2024)
valid: 253,507행 (season=2024)
valid 타겟 평균(r): 0.4861


## 정규화(L2) 튜닝

CatBoost는 리프 값에 대한 **L2만** 지원한다 (`l2_leaf_reg`, 기본값 3.0). L1/L0 페널티 옵션 자체가 없다 — 진짜 L1이 필요하면 LightGBM(`lambda_l1`)이나 XGBoost(`reg_alpha`)를 써야 한다.

기본값(3.0)이 최적인지 확인하려고 단일 OOT 분할(season<2024 학습 / 2024 검증)로 몇 개 값을 스윕했다.

| l2_leaf_reg | Brier | Score |
| --- | --- | --- |
| 3.0 (기본값) | 0.248061 | 698.78 |
| 10.0 | 0.248021 | 714.90 |
| **15.0** | **0.247972** | **734.49** |
| 30.0 | 0.248020 | 715.21 |
| 50.0 | 0.248020 | 715.44 |
| 100.0 | 0.248029 | 711.75 |

15.0 부근에서 국지 최적점이 나와서 채택했다. 다만 이건 단일 분할 기준이라 노이즈가 섞여있을 수 있다 — rolling OOT(2022/2023/2024) 평균으로 재검증하는 걸 다음 단계로 남겨둔다. `eda.py`처럼 이 상수도 `baseline_catboost.py`의 `L2_LEAF_REG`로 빼놨다.

## 학습

`early_stopping_rounds=100`으로 valid BrierScore가 100라운드 동안 개선 안 되면 멈춘다. `use_best_model=True`라 최종 모델은 valid 성능이 가장 좋았던 시점으로 되돌아간다. `l2_leaf_reg=L2_LEAF_REG`(15.0)로 학습.

In [3]:
model = train_catboost(train_df, valid_df)

0:	learn: 0.2495727	test: 0.2498987	best: 0.2498987 (0)	total: 203ms	remaining: 6m 45s


100:	learn: 0.2445101	test: 0.2481202	best: 0.2481202 (100)	total: 11.5s	remaining: 3m 35s


200:	learn: 0.2439739	test: 0.2479844	best: 0.2479827 (195)	total: 22.7s	remaining: 3m 23s


300:	learn: 0.2436681	test: 0.2480017	best: 0.2479721 (203)	total: 33.9s	remaining: 3m 11s


Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.2479721088
bestIteration = 203

Shrink model to first 204 iterations.


## 검증 결과

리더보드 산식(`Score = max(0, 100000 × (1 - Brier/r(1-r)))`)을 그대로 계산해서 로컬 점수를 추정한다. 참고로 Phase 2 수료 기준은 549.51점.

In [4]:
metrics = evaluate(model, valid_df)
for k, v in metrics.items():
    print(f"{k}: {v}")

n: 253507
r (실제 성공률): 0.4861
brier: 0.247972
baseline_brier (r(1-r)): 0.249807
score (리더보드 산식): 734.49
auc: 0.5485
calibration_slope: 1.0902


### 이전 실험 수치와 비교

| 모델 | 2024 Brier | 로컬 Score |
| --- | --- | --- |
| 우리 CatBoost 베이스라인 (피처 엔지니어링 없음, l2_leaf_reg=15) | 0.2480 | 734.49 |
| 이전 실험 E8 (HistGradientBoosting 단일) | 0.2481 | - |
| 이전 실험 E12 (3모델 앙상블 + calibration) | 0.2480 | 711.58 (Public LB 727.72) |

피처 엔지니어링(regime 플래그, EB smoothing 등)을 전혀 안 넣은 첫 시도인데도 이전 실험의 최종 3모델 앙상블(E12)보다 로컬 점수가 높게 나왔다. Phase 2 수료 기준(549.51)은 여유 있게 넘김. calibration_slope가 1.09로 1에 가까워서 보정도 나쁘지 않다. 다만 우리는 단일 시즌(2024) 검증이고 이전 실험은 rolling 가중평균이라 완전히 같은 조건 비교는 아니라는 점은 감안.

## 피처 중요도

CatBoost 자체 `get_feature_importance` (PredictionValuesChange 방식) 기준.

In [5]:
imp = pd.Series(model.get_feature_importance(to_pool(valid_df)), index=FEATURES).sort_values(ascending=False)
imp.head(20)

asof_pitcher_success_rate               11.417579
game_type                               11.400806
pitcher_team_id                          8.401275
asof_pitcher_ball_rate                   7.591048
batter_team_id                           6.955472
asof_pitcher_prev5_game_success_rate     5.579647
asof_pitcher_reverse_rate                5.347660
batter_hand                              4.262082
pitcher_hand                             4.038795
game_month                               3.659701
asof_pitcher_prev1_game_success_rate     2.944471
asof_pitcher_prev3_game_success_rate     2.828534
asof_pitcher_n                           2.665662
asof_batter_n                            2.512707
balls_before                             2.344293
asof_pitcher_offspeed_rate               2.295821
asof_batter_success_rate                 2.152372
asof_pitcher_strike_rate                 1.454433
li                                       1.040269
asof_pitcher_prev1_game_middle_rate      0.921246


EDA 5절의 RandomForest 중요도와 비교하면 흥미로운 차이가 있다. RF에서는 `pitcher_team_id`가 하위권(0.011대)이었는데 CatBoost에서는 3위(9.5)로 훌쩍 올라왔다. 이유로 짐작되는 건, RF는 범주형을 `pd.factorize`로 임의의 정수코드를 매겨서 넣었는데(카테고리 순서에 아무 의미 없음), CatBoost는 카테고리별 타겟 평균(ordered target statistics)으로 인코딩해서 훨씬 효율적으로 팀 정보를 활용한다는 것 — 인코딩 방식 자체가 변수 중요도에 큰 영향을 준다는 걸 다시 확인한 셈이다 (지난번 `season`을 순서형→수치형으로 바꿨을 때 중요도가 뛴 것과 같은 종류의 교훈).

`game_type`(2위)도 여전히 상위권이라 이전 실험이 찾은 regime 신호가 CatBoost에서도 잡히고 있다는 뜻.

## 모델 저장

검증용 모델은 [models/catboost_valid_2024.cbm](models/catboost_valid_2024.cbm)에, **실제 제출용(전체 데이터 재학습) 모델**은 [models/catboost_baseline.cbm](models/catboost_baseline.cbm)에 저장한다. `submit/model/`에 넣는 건 후자.

In [6]:
print("validation model saved:", (MODEL_DIR / "catboost_valid_2024.cbm").exists())
print("final (submission) model saved:", (MODEL_DIR / "catboost_baseline.cbm").exists())

validation model saved: True
final (submission) model saved: True


## 실제 제출 결과 및 문제 발견

위 `catboost_valid_2024.cbm`(2019~2023만 학습)로 제출했더니 **Public LB 662.85점**이 나왔다. 로컬 검증(734.49)보다 71점 넘게 낮다.

원인: 위 학습에서 `season==2024`는 **검증(조기종료 기준)에만 쓰고 학습에는 안 넣었다.** 즉 실제 제출 모델이 2019~2023 데이터(122만행)로만 학습됐고, 가장 최근이자 drift상 가장 중요할 2024년(25만행, 전체의 17%)이 통째로 빠진 채로 2025년 test를 예측한 셈이다. 이전 실험의 로컬-퍼블릭 격차(로컬 711.58 → 퍼블릭 727.72, 오히려 상승)와 반대 방향으로 크게 벌어진 것도 이 문제일 가능성이 높다.

**수정**: 검증으로 적정 학습 라운드 수(`best_iteration`)를 찾은 뒤, 그 라운드 수 그대로 **2019~2024 전체 데이터로 재학습**한 모델을 실제 제출용으로 쓴다. 검증 단계에서 찾은 하이퍼파라미터/라운드 수는 유지하되, 데이터는 최신 시즌까지 전부 포함시키는 표준적인 "최종 재학습(final refit)" 절차.

In [7]:
best_iteration = model.get_best_iteration() + 1
print(f"검증에서 찾은 최적 iteration: {best_iteration}")

final_model = train_final_full(df, best_iteration)

검증에서 찾은 최적 iteration: 204


0:	learn: 0.2496315	total: 176ms	remaining: 35.7s


100:	learn: 0.2452131	total: 15.4s	remaining: 15.7s


200:	learn: 0.2446678	total: 30.1s	remaining: 450ms


203:	learn: 0.2446615	total: 30.6s	remaining: 0us


이 `final_model`은 2024년을 포함해서 학습했기 때문에 2024년에 대한 held-out 검증 점수를 낼 수 없다 (이미 학습에 쓴 데이터라 채점이 무의미함). 그래서 로컬 Brier/Score는 위 `catboost_valid_2024.cbm` 결과(0.247972 / 734.49)를 계속 참고용으로 쓰고, 이 `final_model`이 실제 제출용이다. 2025년 실제 성능은 다음 제출 결과로 확인해야 한다.

## 다음 후보

- `season≥2023 × game_type=F` 같은 regime 교호작용 피처 명시적으로 추가 (이전 실험이 가장 큰 단일 개선을 봤던 지점)
- rolling OOT를 2022/2023/2024 세 시즌 가중평균으로 확장 (지금은 2024 단일 검증만)
- LightGBM도 같은 방식으로 적합해서 CatBoost와 블렌드

## 변수 간 교호작용 탐색 (CatBoost 네이티브)

[elastic_net.ipynb](elastic_net.ipynb)에서 `game_type`을 손으로 찾은 교호작용(`× season_regime`)으로 고쳐봤는데, 그래도 CatBoost(789.23)에는 한참 못 미쳤다(385.29). "선형모델이 놓치고 있는 다른 교호작용이 있을 것"이라는 추측을 CatBoost의 트리 구조에서 직접 확인해본다.

`get_feature_importance(type="Interaction")`는 SHAP interaction values보다 훨씬 가볍다 — 데이터 샘플을 다시 돌릴 필요 없이 이미 학습된 트리의 분할 구조에서 바로 "어떤 변수 쌍이 같은 트리 경로에 자주 같이 등장하는가"를 계산한다.

In [ ]:
top_interactions = feature_interaction_ranking(final_model, top_n=25)
top_interactions

### 발견

**`game_type`은 상위 25개에 아예 없다.** 대신 가장 강한 교호작용은 `season`이 여러 변수와 광범위하게 얽혀 있는 것이다:

- `asof_pitcher_success_rate × season` — 1위
- `batter_team_id × season`, `asof_pitcher_reverse_rate × season`, `asof_batter_success_rate × season`, `asof_pitcher_prev1/3/5_game_success_rate × season` — 전부 상위권

그 외에 `pitcher_team_id × batter_team_id`(매치업 효과), `pitcher_team_id × game_month`도 꽤 강하다.

**해석**: 리그 전체 성공률이 매년 계속 떨어지고 있으니(연도별 drift), "같은 55% 성공률"이라는 숫자가 갖는 의미 자체가 연도마다 다르다 — 2019년의 55%는 평범한 수준이지만 2024년의 55%는 꽤 좋은 축일 수 있다. 이게 `game_type` 하나만의 문제가 아니라 **거의 모든 `asof_*` rate 피처가 `season`과 교호작용을 갖는** 더 넓은 패턴이라는 뜻이다.

**Elastic Net과의 격차 설명**: 선형모델로 이걸 다 따라잡으려면 `season × asof_pitcher_success_rate`, `season × asof_pitcher_reverse_rate`, `pitcher_team_id × batter_team_id` 등을 전부 손으로 교호작용 변수로 만들어줘야 한다. `game_type` 하나 고치는 데도 파라미터가 늘어나서 오히려 손해였는데(교호작용 336점 < 완전제외 385점), 이런 교호작용을 수십 개 다 만들면 과적합 위험이 커진다. 트리는 이걸 전부 공짜로, 필요한 만큼만 골라서 쓴다 — 이게 385 vs 789 격차의 실제 원인으로 보인다.